# Exercise: Implementing Causal Self-Attention

Welcome! In this exercise, you'll get hands-on experience implementing the Causal Self-Attention mechanism, a core component of decoder-style transformer models like GPT. You'll build this layer from scratch using PyTorch.

By the end of this notebook, you will be able to:
*   Implement the query, key, and value projection layers.
*   Calculate scaled dot-product attention scores.
*   Create and apply a causal mask to ensure the model can't "see" into the future.

Let's get started!

In [ ]:
import torch
import torch.nn as nn
import math

# Set a seed for reproducibility
torch.manual_seed(42)

class Config:
    """Configuration class for model parameters."""
    def __init__(self):
        # The dimensionality of the model's embeddings
        self.n_embd = 64
        # The number of attention heads. For this exercise, we'll keep it simple with 1.
        self.n_head = 1
        # The maximum sequence length the model can handle
        self.block_size = 128
        # Dropout probability
        self.dropout = 0.1

config = Config()

# Let's ensure the embedding dimension is divisible by the number of heads
assert config.n_embd % config.n_head == 0, "n_embd must be divisible by n_head"

print("Setup complete. Your PyTorch version is:", torch.__version__)

### Exercise 1: Implement the `CausalSelfAttention` Class

Your first task is to implement the `CausalSelfAttention` module. This involves setting up the projection layers in the constructor and then implementing the complete forward pass logic.

**Instructions:**
1.  In the `__init__` method, define the linear layers for query (`q_proj`), key (`k_proj`), value (`v_proj`), and the final output projection (`c_proj`).
2.  In the `forward` method, implement the following steps:
    *   Get the batch size (B), sequence length (T), and embedding dimension (C) from the input tensor `x`.
    *   Project `x` into query, key, and value tensors using the layers you defined.
    *   Calculate the attention scores by taking the dot product of queries and keys. Don't forget to scale by `1 / sqrt(key_dimension)`.
    *   **Apply the causal mask**: This is the most important step for a generative model! Create a lower-triangular matrix using `torch.tril` and use it to mask the upper-triangular portion of the attention scores. A common way to do this is to set the masked values to a very large negative number (like `-1e9` or `-float('inf')`) before the softmax step.
    *   Apply the softmax function to the masked scores to get the attention weights.
    *   Apply the attention weights to the value tensor.
    *   Pass the result through the final output projection layer.

**Hint:** You can create the causal mask on-the-fly inside the `forward` pass. A common pattern is `torch.tril(torch.ones(T, T))`. You can use the `masked_fill_` method to apply the mask.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_embd = config.n_embd
        self.n_head = config.n_head
        
        # We will use a single large projection for Q, K, V for efficiency
        # and then split the result.
        ### START CODE HERE ###
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        # Output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        ### END CODE HERE ###
        
        # A buffer is a tensor that is part of the model's state but is not a parameter.
        # It is not updated by the optimizer. Here we use it for the causal mask.
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                     .view(1, 1, config.block_size, config.block_size))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Performs the forward pass for causal self-attention.
        
        Args:
            x (torch.Tensor): Input tensor of shape (B, T, C)
                              B = batch size, T = sequence length, C = embedding dim
                              
        Returns:
            torch.Tensor: Output tensor of the same shape as input (B, T, C)
        """
        # Batch size, sequence length, embedding dimensionality
        B, T, C = x.size() 
        
        ### START CODE HERE ###
        # 1. Project to Q, K, V from a single linear layer
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)

        # For this exercise, we are using n_head=1, so we don't need to reshape for multi-head.
        # We'll just add the head dimension for consistency.
        # (B, T, C) -> (B, n_head, T, head_size) where n_head=1, head_size=C
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        
        # 2. Calculate attention scores ("affinities")
        # (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        
        # 3. Apply the causal mask
        # self.bias is pre-computed up to block_size. We slice it for the current sequence length T.
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        
        # 4. Apply softmax to get weights
        att = F.softmax(att, dim=-1)
        
        # 5. Apply weights to values
        y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        
        # 6. Concatenate heads and project output
        # Here, with n_head=1, this is just a reshape.
        y = y.transpose(1, 2).contiguous().view(B, T, C) 
        y = self.c_proj(y)
        ### END CODE HERE ###
        
        return y

In [ ]:
# Test cell for Exercise 1
import torch.nn.functional as F

print("--- Running Test for CausalSelfAttention ---")
# Instantiate the model
attention_layer = CausalSelfAttention(config)

# Create a dummy input tensor
B, T, C = 2, 5, config.n_embd
dummy_x = torch.randn(B, T, C)

# Perform a forward pass
output = attention_layer(dummy_x)

# 1. Test output shape
expected_shape = (B, T, C)
assert output.shape == expected_shape, f"Output shape is incorrect. Got {output.shape}, expected {expected_shape}"
print("✅ Test 1/2: Output shape is correct.")

# 2. Test the causal nature of the attention weights
# We can access the intermediate attention weights by slightly modifying the forward pass
# This is for testing purposes only.
def get_attention_weights(model, x):
    B, T, C = x.size()
    q, k, v = model.c_attn(x).split(model.n_embd, dim=2)
    k = k.view(B, T, model.n_head, C // model.n_head).transpose(1, 2)
    q = q.view(B, T, model.n_head, C // model.n_head).transpose(1, 2)
    att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
    att = att.masked_fill(model.bias[:,:,:T,:T] == 0, float('-inf'))
    att = F.softmax(att, dim=-1)
    return att

# Get the weights for the dummy input
weights = get_attention_weights(attention_layer, dummy_x)
# The weights matrix should be lower-triangular
# We check if the sum of the upper triangle (excluding the diagonal) is zero.
is_causal = torch.all(torch.triu(weights[0, 0], diagonal=1) == 0)
assert is_causal, "Attention is not causal! The model is looking at future tokens."
print("✅ Test 2/2: Causal mask is applied correctly.")

print("\n🎉 All tests passed for CausalSelfAttention! Great work!")

### Exercise 2: A Closer Look at Scaled Dot-Product Attention

Excellent! You've implemented the full `CausalSelfAttention` module. Now, let's break it down and implement the core mathematical operation as a standalone function. This is often how it's implemented in libraries like PyTorch for modularity.

**Instructions:**
You will implement a function `scaled_dot_product_attention` that takes queries, keys, values, and an optional mask as input.

*   Calculate the dot product of Q and K transpose.
*   Scale the scores by `1 / sqrt(d_k)`, where `d_k` is the dimension of the keys.
*   If a `mask` is provided, apply it to the scores. Fill the masked positions with `-inf`.
*   Apply softmax to get the attention weights.
*   Return the weighted sum of the values and the attention weights themselves.

In [ ]:
def scaled_dot_product_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, mask: torch.Tensor = None) -> (torch.Tensor, torch.Tensor):
    """
    Calculates the scaled dot-product attention.
    
    Args:
        q (torch.Tensor): Query tensor; shape (B, nh, T, hs)
        k (torch.Tensor): Key tensor; shape (B, nh, T, hs)
        v (torch.Tensor): Value tensor; shape (B, nh, T, hs)
        mask (torch.Tensor, optional): Boolean mask; shape (T, T). Defaults to None.
        
    Returns:
        (torch.Tensor, torch.Tensor): A tuple containing the output and the attention weights.
    """
    # Get the dimension of the key vector
    d_k = q.size(-1)
    
    # Get the dimension of the key vector
    d_k = q.size(-1)
    
    # 1. Calculate scores
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(d_k)
    
    # 2. Apply mask if provided
    if mask is not None:
        # The mask shape is (T, T), but scores is (B, nh, T, T).
        # PyTorch will broadcast the mask correctly.
        scores = scores.masked_fill(mask == 0, float('-inf'))
        
    # 3. Apply softmax
    weights = F.softmax(scores, dim=-1)
    
    # 4. Get the weighted values
    output = weights @ v
    
    return output, weights
    
    return output, weights

In [ ]:
# Test cell for Exercise 2
print("--- Running Test for scaled_dot_product_attention ---")

# Setup dummy tensors
B, nh, T, hs = 2, 1, 4, config.n_embd
q = torch.randn(B, nh, T, hs)
k = torch.randn(B, nh, T, hs)
v = torch.randn(B, nh, T, hs)
mask = torch.tril(torch.ones(T, T))

# Function call
output, weights = scaled_dot_product_attention(q, k, v, mask=mask)

# 1. Test output shapes
expected_output_shape = (B, nh, T, hs)
expected_weights_shape = (B, nh, T, T)
assert output.shape == expected_output_shape, f"Output shape is incorrect. Got {output.shape}, expected {expected_output_shape}"
assert weights.shape == expected_weights_shape, f"Weights shape is incorrect. Got {weights.shape}, expected {expected_weights_shape}"
print("✅ Test 1/2: Output and weights shapes are correct.")

# 2. Test masking
is_causal = torch.all(torch.triu(weights[0, 0], diagonal=1) == 0)
assert is_causal, "Mask was not applied correctly. Upper triangle of weights should be zero."
print("✅ Test 2/2: Masking is applied correctly.")

print("\n🎉 All tests passed for scaled_dot_product_attention! You're doing great!")

### Exercise 3: Putting It All Together

Fantastic work! Now you have a modular, reusable `scaled_dot_product_attention` function. The final step is to refactor your original `CausalSelfAttention` class to use this new function. This is a common practice that makes code cleaner and easier to debug.

**Instructions:**
Rewrite the `forward` method of the `CausalSelfAttentionRefactored` class below. This time, instead of manually calculating scores, applying the mask, and multiplying by V, you should call the `scaled_dot_product_attention` function you just wrote.

In [ ]:
class CausalSelfAttentionRefactored(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_embd = config.n_embd
        self.n_head = config.n_head
        
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        
        # We still need the bias for the mask, to pass it to our helper function
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                     .view(1, 1, config.block_size, config.block_size))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.size()
        
        ### START CODE HERE ###
        # 1. Project to Q, K, V and reshape for multi-head attention
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        
        # 2. Call the scaled_dot_product_attention function
        # Slice the mask to the appropriate size for the current sequence length T
        mask = self.bias[:, :, :T, :T]
        y, _ = scaled_dot_product_attention(q, k, v, mask=mask) # We don't need the weights here
        
        # 3. Reshape and project the output
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)
        ### END CODE HERE ###
        
        return y

In [ ]:
# Integration Test: Wires Exercise 1 and 2 together
print("--- Running Integration Test for Refactored Class ---")

# To ensure a fair comparison, we'll create two models
# and load the weights from the first into the second.
model_v1 = CausalSelfAttention(config)
model_v2_refactored = CausalSelfAttentionRefactored(config)

# Copy weights to ensure they are identical
model_v2_refactored.c_attn.load_state_dict(model_v1.c_attn.state_dict())
model_v2_refactored.c_proj.load_state_dict(model_v1.c_proj.state_dict())

# Create the same dummy input
B, T, C = 2, 8, config.n_embd
dummy_x = torch.randn(B, T, C)

# Get outputs from both models
output_v1 = model_v1(dummy_x)
output_v2 = model_v2_refactored(dummy_x)

# 1. Test output shape
expected_shape = (B, T, C)
assert output_v2.shape == expected_shape, f"Refactored output shape is incorrect. Got {output_v2.shape}, expected {expected_shape}"
print("✅ Test 1/2: Refactored output shape is correct.")

# 2. Test that outputs are identical (or very close due to float precision)
assert torch.allclose(output_v1, output_v2, atol=1e-6), "Outputs of original and refactored models do not match!"
print("✅ Test 2/2: Refactored model produces the same output as the original.")

print("\n🎉 Congratulations! You have successfully implemented and refactored a causal self-attention layer.")
print("This is a critical building block for modern language models.")